# Reproducible execution: association rules in open CAR data

This notebook documents the public-data execution used in the article. It uses only municipal CSV files downloaded from the official Consulta Pública do Cadastro Ambiental Rural portal. The analytical unit is one CAR rural property identified by `cod_imovel`. No Trino, Iceberg, GOLD, SNCR, SIGEF, or restricted institutional database is used.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd

INPUT_DIR = Path(r'D:\USUARIO_CELIO\ARTIGO FINAL MINERAÇÃO DE DADOS\DADOS_ABERTOS_CAR_2026-09-21_COMUNS')
OUTPUT_DIR = Path(r'D:\USUARIO_CELIO\ARTIGO FINAL MINERAÇÃO DE DADOS\RESULTADOS_PUBLICOS_ARTIGO_EXECUTADO_2026-09-21')
PIPELINE = Path(r'D:\USUARIO_CELIO\Documentos\ChatGPT\MESTRADO\pipeline_dados_abertos_car.py')
AUDIT = Path(r'D:\USUARIO_CELIO\Documentos\ChatGPT\MESTRADO\auditar_regras_car_aberto.py')
assert INPUT_DIR.exists(), INPUT_DIR
assert PIPELINE.exists(), PIPELINE
assert AUDIT.exists(), AUDIT
print('Input:', INPUT_DIR)
print('Output:', OUTPUT_DIR)

In [ ]:
csv_files = sorted(INPUT_DIR.glob('*.csv'))
print('CSV files:', len(csv_files))
for path in csv_files:
    print(path.name)
assert any('AREA_IMOVEL' in p.name.upper() for p in csv_files)
assert not any(token in str(p).upper() for p in csv_files for token in ['TRINO', 'ICEBERG', 'GOLD'])

In [ ]:
cmd = [sys.executable, str(PIPELINE), str(INPUT_DIR), str(OUTPUT_DIR), '--min-support', '0.01', '--min-confidence', '0.60', '--max-len', '3']
run = subprocess.run(cmd, check=True, capture_output=True, text=True)
print(run.stdout)
audit_run = subprocess.run([sys.executable, str(AUDIT), str(OUTPUT_DIR)], check=True, capture_output=True, text=True)
print(audit_run.stdout)

In [ ]:
summary = json.loads((OUTPUT_DIR / 'resumo_execucao_car_aberto.json').read_text(encoding='utf-8'))
audit = json.loads((OUTPUT_DIR / 'resumo_auditoria_regras.json').read_text(encoding='utf-8'))
print(json.dumps(summary, ensure_ascii=False, indent=2))
print(json.dumps(audit, ensure_ascii=False, indent=2))
assert summary['total_transacoes'] == 11340
assert summary['apriori_eclat_coincidem'] is True
assert audit['regras_candidatas_artigo'] == 12

In [ ]:
candidates = pd.read_csv(OUTPUT_DIR / 'regras_candidatas_artigo.csv', sep=';')
display(candidates)
print('Candidate rules:', len(candidates))
print('Notebook validation: OK')